In [ ]:
!pip install transformers datasets scikit-learn torch accelerate -q

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train1.csv to train1 (1).csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("train1.csv")

# Label encoding
labels = sorted(df["class_label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["class_label"].map(label2id)

print("Classes:", label2id)
print("Total samples:", len(df))

# Split: 60 train, 20 val, 20 test — stratified to keep class balance
train_df, temp_df = train_test_split(df, test_size=0.40, stratify=df["label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Classes: {'account_block_scam': 0, 'benign': 1, 'fake_payment_portal': 2, 'impersonation': 3, 'kyc_scam': 4, 'phishing_link': 5}
Total samples: 14014
Train: 8408 | Val: 2803 | Test: 2803


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=256,   # increased from 128
        return_tensors="pt"
    )

train_enc = tokenize(train_df["message_text"])
val_enc   = tokenize(val_df["message_text"])
test_enc  = tokenize(test_df["message_text"])

In [ ]:
import torch

class SMSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = SMSDataset(train_enc, train_df["label"].values)
val_dataset   = SMSDataset(val_enc,   val_df["label"].values)
test_dataset  = SMSDataset(test_enc,  test_df["label"].values)

In [ ]:
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Class weights so rare classes aren't ignored
class_weights = compute_class_weight("balanced", classes=np.arange(len(label2id)), y=train_df["label"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", dict(zip(labels, class_weights.numpy().round(2))))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Class weights: {'account_block_scam': np.float32(1.06), 'benign': np.float32(0.78), 'fake_payment_portal': np.float32(1.06), 'impersonation': np.float32(1.06), 'kyc_scam': np.float32(1.06), 'phishing_link': np.float32(1.06)}


In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score
import torch
import numpy as np

# 🔹 Load model again (IMPORTANT)
MODEL_NAME = "google/muril-base-cased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 🔹 Weighted Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

# 🔹 Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# 🔹 Training Arguments
args = TrainingArguments(
    output_dir="./muril-sms-fraud",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=True,
)

# 🔹 Trainer
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 🔹 Train
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Epoch,Training Loss,Validation Loss,Accuracy
1,1.161250,1.041908,0.859793
2,0.639436,0.526630,0.902248
3,0.411265,0.397112,0.900464
4,0.362538,0.307305,0.917232
5,0.294276,0.269545,0.914377
6,0.216592,0.260419,0.917945
7,0.204676,0.265690,0.916161
8,0.188530,0.268067,0.919372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4208, training_loss=0.5026159410297644, metrics={'train_runtime': 2356.5984, 'train_samples_per_second': 28.543, 'train_steps_per_second': 1.786, 'total_flos': 8849268818509824.0, 'train_loss': 0.5026159410297644, 'epoch': 8.0})

In [ ]:
trainer.save_model("./muril-sms-fraud-final")
tokenizer.save_pretrained("./muril-sms-fraud-final")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./muril-sms-fraud-final",
    tokenizer="./muril-sms-fraud-final",
    device=0  # GPU, use -1 for CPU
)

def predict(message):
    result = classifier(message, truncation=True, max_length=128)[0]
    label = result["label"]
    confidence = round(result["score"] * 100, 2)

    # Risk level
    if label == "benign":
        risk = "✅ SAFE"
    elif confidence < 70:
        risk = "⚠️ SUSPICIOUS"
    else:
        risk = "🚨 FRAUD"

    print(f"Message : {message}")
    print(f"Label   : {label}")
    print(f"Risk    : {risk}")
    print(f"Confidence: {confidence}%")
    print("-" * 60)

# Test with sample messages
predict("ਕੀ ਤੁਸੀਂ ਆਪਣੇ ਨਾਮ ਤੇ ਕਿੰਨੇ ਸਿਮ ਹਨ ਜਾਣਨਾ ਚਾਹੁੰਦੇ ਹੋ? ਇਹ ਦੇਖਣ ਲਈ ਸੰਚਾਰ ਸਾਥੀ ਡਾਊਨਲੋਡ ਕਰੋ : ਐਂਡਰਾਇਡ : Android : https://t.jio/ISATHI/MWxHL9 iOS : https://t.jio/ISATHI/ios ਦੂਰਸੰਚਾਰ ਵਿਭਾਗ, ਭਾਰਤ ਸਰਕਾਰ")
predict("UPI Activated! You've linked your Bank A/c to UPI on Google Pay. NEVER share OTP, PIN, card info. - HDFC Bank")
predict("Ekart Update: Martucci Heavy Unisex for C... will be delivered by 11pm today. Track: http://fkrt.it/EKARTL/YsOfy5NNNN")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Message : ਕੀ ਤੁਸੀਂ ਆਪਣੇ ਨਾਮ ਤੇ ਕਿੰਨੇ ਸਿਮ ਹਨ ਜਾਣਨਾ ਚਾਹੁੰਦੇ ਹੋ? ਇਹ ਦੇਖਣ ਲਈ ਸੰਚਾਰ ਸਾਥੀ ਡਾਊਨਲੋਡ ਕਰੋ : ਐਂਡਰਾਇਡ : Android : https://t.jio/ISATHI/MWxHL9 iOS : https://t.jio/ISATHI/ios ਦੂਰਸੰਚਾਰ ਵਿਭਾਗ, ਭਾਰਤ ਸਰਕਾਰ
Label   : kyc_scam
Risk    : ⚠️ SUSPICIOUS
Confidence: 66.71%
------------------------------------------------------------
Message : UPI Activated! You've linked your Bank A/c to UPI on Google Pay. NEVER share OTP, PIN, card info. - HDFC Bank
Label   : benign
Risk    : ✅ SAFE
Confidence: 95.78%
------------------------------------------------------------
Message : Ekart Update: Martucci Heavy Unisex for C... will be delivered by 11pm today. Track: http://fkrt.it/EKARTL/YsOfy5NNNN
Label   : fake_payment_portal
Risk    : 🚨 FRAUD
Confidence: 96.69%
------------------------------------------------------------


In [ ]:
sneaky_messages = [
    # --- Utility / Electricity ---
    "Your electricity meter reading for November is due. Submit at: https://msedcl-meter-update.in/submit",
    "Power outage scheduled in your area on 25 Nov 6AM-2PM. Confirm your address: https://bescom-schedule.in/verify",
    "Smart meter upgrade pending for your connection. Book slot: https://tatapower-smartmeter.in/book",
    "Your electricity subsidy of Rs.850 is ready. Link your Aadhaar to receive: https://discom-subsidy.in/claim",
    "Meter reading could not be taken. Estimated bill generated. Dispute at: https://kseb-billing.in/dispute",

    # --- Water / Gas ---
    "Jal Jeevan Mission: Your household water connection application is pending approval. Complete KYC: https://jaljeevan-apply.in/kyc",
    "Gas cylinder price revised. Update your bank details for subsidy continuation: https://indane-subsidy.in/update",
    "BWSSB water audit team will visit your premises. Confirm appointment: https://bwssb-audit.in/confirm",
    "Your Piped Natural Gas connection meter test is scheduled. Approve visit: https://mahanagargas-test.in/approve",
    "Water tank cleaning drive in your society. Register your flat: https://mcgm-cleaning.in/register",

    # --- Bank / Finance ---
    "Your FD of Rs.1,20,000 is maturing on 30 Nov. Renew or withdraw at: https://sbi-fd-renew.in/options",
    "Income Tax refund of Rs.4,200 approved. Credit to account ending 7823. Verify: https://itax-refund.in/verify",
    "Your loan EMI bounce charge of Rs.500 has been waived. Confirm acknowledgement: https://hdfcloan-waiver.in/ack",
    "UPI limit upgrade approved for your account. Activate now: https://npci-upi-limit.in/activate",
    "Your credit card reward points (3,240 pts) expire on 30 Nov. Redeem: https://axisbank-rewards.in/redeem",

    # --- KYC / Account ---
    "As per RBI guidelines, video KYC is mandatory for accounts opened before 2022. Complete: https://rbi-vkyc.in/start",
    "Your nominee details are incomplete as per new SEBI rules. Update: https://sebi-nominee.in/update",
    "PAN-Aadhaar linking deadline extended to Dec 31. Link now to avoid Rs.1000 penalty: https://pan-aadhaar-link.in/now",
    "Your mobile number is not linked to your bank account. Link to avoid transaction limits: https://unionbank-mobile.in/link",
    "CIBIL score update: New inquiry detected on your report. Review: https://cibil-alert.in/review",

    # --- Delivery / eCommerce ---
    "Your Amazon order #402-8847234 could not be delivered. Reschedule: https://amzn-redeliver.in/slot",
    "Customs duty of Rs.340 pending for your international shipment. Pay to release: https://fedex-customs.in/pay",
    "Your Flipkart return pickup is scheduled for tomorrow 10AM-1PM. Confirm: https://fk-pickup.in/confirm",
    "Package held at courier facility due to incomplete address. Update: https://bluedart-address.in/update",
    "Your DTDC parcel requires re-verification due to mismatch. Verify: https://dtdc-verify.in/check",

    # --- Government / Official ---
    "PM Awas Yojana: Your application status updated. Check allotment letter: https://pmay-status.in/letter",
    "Ration card digitization: Link your Aadhaar to avoid suspension: https://nfsa-ration.in/link",
    "Your Ayushman Bharat card is ready for download. Verify OTP: https://pmjay-card.in/download",
    "Driving licence renewal reminder: Your DL expires on 15 Dec. Renew online: https://sarathi-renew.in/apply",
    "Your passport application (File No. IN2024XXXXXX) requires additional document. Upload: https://passportindia-docs.in/upload",

    # --- Job / HR ---
    "Your PF withdrawal request of Rs.45,000 is under process. Track: https://epfo-track.in/status",
    "New EPF circular: Activate your UAN to continue receiving employer contributions: https://epfo-uan.in/activate",
    "Your Form 16 for AY 2024-25 is ready. Download from: https://traces-form16.in/download",
    "Background verification pending for your job application at Infosys. Complete: https://bgv-infosys.in/verify",
    "Your salary account shows inactive status. Reactivate within 7 days: https://hdfcsalary-reactivate.in/now",

    # --- Insurance ---
    "Your LIC policy premium of Rs.8,240 is due on 5 Dec. Pay to avoid lapse: https://lic-premium.in/pay",
    "Mediclaim cashless approval for your pending claim (Ref: CLM2024XXXX). Confirm: https://starhealth-claim.in/confirm",
    "Your vehicle insurance expires in 7 days. Renew now to avoid fine: https://irdai-renew.in/vehicle",
    "Term plan nominee update pending. Update within 15 days: https://maxlife-nominee.in/update",
    "Health insurance free annual checkup benefit expiring Dec 31. Book: https://niva-checkup.in/book",

    # --- Telecom ---
    "Your Jio number will be deactivated due to incomplete re-verification. Verify: https://jio-reverify.in/now",
    "TRAI new regulation: Submit your telecom KYC by Dec 15 to avoid outgoing call block: https://trai-kyc.in/submit",
    "Your Airtel postpaid bill of Rs.1,249 is overdue. Pay to avoid disconnection: https://airtel-bill.in/pay",
    "Vi SIM upgrade to 5G available in your area. Upgrade free: https://vi-5g-upgrade.in/book",
    "Your BSNL broadband speed upgrade request is approved. Confirm activation: https://bsnl-upgrade.in/activate",

    # --- Investments ---
    "Your Zerodha account shows unsettled funds of Rs.12,400. Settle to avoid penalty: https://zerodha-settle.in/now",
    "Mutual fund SIP mandate rejection: Update bank details to continue SIP: https://groww-mandate.in/update",
    "Your Sovereign Gold Bond application (Tranche 4) allotment is confirmed. Download: https://rbi-sgb.in/allotment",
    "NSE trading halt due to KYC mismatch on your account. Resolve: https://nse-kyc.in/resolve",
    "Your PPF account passbook update is pending. Update at: https://indiapost-ppf.in/update",
]

print(f"Total messages: {len(sneaky_messages)}")
print("=" * 60)

for i, msg in enumerate(sneaky_messages, 1):
    result = classifier(msg, truncation=True, max_length=128)[0]
    label = result["label"]
    confidence = round(result["score"] * 100, 2)

    if label == "benign":
        risk = "✅ SAFE (MISSED!)"
    elif confidence < 70:
        risk = "⚠️ SUSPICIOUS"
    else:
        risk = "🚨 CAUGHT"

    print(f"[{i:02d}] {risk} | {label} ({confidence}%)")
    print(f"      {msg[:80]}...")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Total messages: 50
[01] ⚠️ SUSPICIOUS | impersonation (39.42%)
      Your electricity meter reading for November is due. Submit at: https://msedcl-me...

[02] 🚨 CAUGHT | impersonation (79.96%)
      Power outage scheduled in your area on 25 Nov 6AM-2PM. Confirm your address: htt...

[03] ⚠️ SUSPICIOUS | impersonation (50.21%)
      Smart meter upgrade pending for your connection. Book slot: https://tatapower-sm...

[04] ✅ SAFE (MISSED!) | benign (95.82%)
      Your electricity subsidy of Rs.850 is ready. Link your Aadhaar to receive: https...

[05] 🚨 CAUGHT | account_block_scam (79.04%)
      Meter reading could not be taken. Estimated bill generated. Dispute at: https://...

[06] 🚨 CAUGHT | kyc_scam (95.84%)
      Jal Jeevan Mission: Your household water connection application is pending appro...

[07] 🚨 CAUGHT | impersonation (92.97%)
      Gas cylinder price revised. Update your bank details for subsidy continuation: h...

[08] 🚨 CAUGHT | impersonation (91.95%)
      BWSSB water aud

In [ ]:
import re
import urllib.parse

SAFE_DOMAINS = {
    "gov.in", "nic.in", "india.gov.in", "mygov.in", "digilocker.gov.in",
    "incometax.gov.in", "epfindia.gov.in", "passportindia.gov.in",
    "trai.gov.in", "dot.gov.in", "uidai.gov.in",
    "sbi.co.in", "hdfcbank.com", "icicibank.com", "axisbank.com",
    "bankofbaroda.in", "pnbindia.in", "unionbankofindia.co.in",
    "kotak.com", "yesbank.in", "idfcfirstbank.com",
    "jio.com", "airtel.in", "airtel.com", "vi.in", "bsnl.co.in",
    "flipkart.com", "amazon.in", "amazon.com", "myntra.com", "meesho.com",
    "fkrt.it", "paytm.com", "phonepe.com", "gpay.app", "upi.npci.org.in",
    "razorpay.com", "billdesk.com", "licindia.in", "icicilombard.com",
    "hdfclife.com", "sbigeneral.in", "zerodha.com", "groww.in",
    "bluedart.com", "delhivery.com", "dtdc.com", "indiapost.gov.in",
    "ekartlogistics.com", "t.jio.com", "bit.ly", "tiny.cc",
}

SUSPICIOUS_PATTERNS = [
    r'\d{4,}',
    r'(secure|verify|update|login|kyc|pay|claim|confirm)-\w+\.(in|com|net|org)',
    r'\w+-(in|com)\.\w+',
    r'[a-z]{3,}-[a-z]{3,}-[a-z]{3,}',
]

def extract_urls(text):
    return re.findall(r'https?://[^\s]+|www\.[^\s]+', text)

def get_root_domain(url):
    try:
        parsed = urllib.parse.urlparse(url if url.startswith('http') else 'http://' + url)
        parts = parsed.netloc.split('.')
        if len(parts) >= 3 and parts[-2] in ('co', 'gov', 'net', 'org'):
            return '.'.join(parts[-3:])
        return '.'.join(parts[-2:])
    except:
        return ""

def check_url(url):
    root = get_root_domain(url)
    if root in SAFE_DOMAINS or any(s in url for s in SAFE_DOMAINS):
        return "SAFE", f"Trusted domain: {root}"
    for pattern in SUSPICIOUS_PATTERNS:
        if re.search(pattern, url, re.IGNORECASE):
            return "DANGEROUS", f"Suspicious pattern in URL"
    return "UNKNOWN", f"Unverified domain: {root}"

def full_analyze(message):
    print(f"📩 Message: {message[:80]}...")
    result = classifier(message, truncation=True, max_length=128)[0]
    nlp_label = result["label"]
    nlp_conf  = round(result["score"] * 100, 2)
    urls = extract_urls(message)
    url_verdicts = []
    for url in urls:
        status, reason = check_url(url)
        url_verdicts.append((url, status, reason))
    has_dangerous_url = any(v[1] == "DANGEROUS" for v in url_verdicts)
    has_unknown_url   = any(v[1] == "UNKNOWN"   for v in url_verdicts)
    has_safe_url      = any(v[1] == "SAFE"       for v in url_verdicts)
    if has_safe_url and nlp_label == "benign":
        final = "✅ SAFE"
    elif has_dangerous_url:
        final = "🚨 FRAUD — Malicious URL detected"
    elif nlp_label != "benign" and nlp_conf >= 85:
        final = "🚨 FRAUD — AI detected scam pattern"
    elif has_unknown_url and nlp_label != "benign":
        final = "🚨 FRAUD — Suspicious URL + AI flagged"
    elif has_unknown_url or (nlp_label != "benign" and nlp_conf >= 60):
        final = "⚠️ SUSPICIOUS — Manual review advised"
    else:
        final = "✅ SAFE"
    print(f"🤖 NLP     : {nlp_label} ({nlp_conf}%)")
    if url_verdicts:
        for url, status, reason in url_verdicts:
            icon = "✅" if status=="SAFE" else "🚨" if status=="DANGEROUS" else "⚠️"
            print(f"🔗 URL     : {icon} {status} → {reason}")
            print(f"           {url[:60]}")
    else:
        print("🔗 URL     : No links found")
    print(f"📊 VERDICT : {final}")
    print("-" * 65)

In [ ]:
full_analyze("ਕੀ ਤੁਸੀਂ ਆਪਣੇ ਨਾਮ ਤੇ ਕਿੰਨੇ ਸਿਮ ਹਨ ਜਾਣਨਾ ਚਾਹੁੰਦੇ ਹੋ? Android : https://t.jio/ISATHI/MWxHL9")
full_analyze("UPI Activated! You've linked your Bank A/c to UPI on Google Pay. NEVER share OTP, PIN, card info. - HDFC Bank")
full_analyze("Ekart Update: will be delivered by 11pm today. Track: http://fkrts.it/EKARTL/YsOfy5NNNN")
full_analyze("Your SBI account is blocked. Verify KYC at: https://sbi-secure-verify-login.in/kyc/update")

📩 Message: ਕੀ ਤੁਸੀਂ ਆਪਣੇ ਨਾਮ ਤੇ ਕਿੰਨੇ ਸਿਮ ਹਨ ਜਾਣਨਾ ਚਾਹੁੰਦੇ ਹੋ? Android : https://t.jio/ISAT...
🤖 NLP     : fake_payment_portal (93.64%)
🔗 URL     : ⚠️ UNKNOWN → Unverified domain: t.jio
           https://t.jio/ISATHI/MWxHL9
📊 VERDICT : 🚨 FRAUD — AI detected scam pattern
-----------------------------------------------------------------
📩 Message: UPI Activated! You've linked your Bank A/c to UPI on Google Pay. NEVER share OTP...
🤖 NLP     : benign (95.78%)
🔗 URL     : No links found
📊 VERDICT : ✅ SAFE
-----------------------------------------------------------------
📩 Message: Ekart Update: will be delivered by 11pm today. Track: http://fkrts.it/EKARTL/YsO...
🤖 NLP     : benign (40.28%)
🔗 URL     : ⚠️ UNKNOWN → Unverified domain: fkrts.it
           http://fkrts.it/EKARTL/YsOfy5NNNN
📊 VERDICT : ⚠️ SUSPICIOUS — Manual review advised
-----------------------------------------------------------------
📩 Message: Your SBI account is blocked. Verify KYC at: https://sbi-secure-verify-login.in/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainer.save_model("/content/drive/MyDrive/muril-sms-fraud-final")
tokenizer.save_pretrained("/content/drive/MyDrive/muril-sms-fraud-final")
print("✅ Model saved to Google Drive!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Google Drive!


In [ ]:
import os

# Check karo kya save hua
for f in os.listdir("/content/drive/MyDrive/muril-sms-fraud-final"):
    print(f)

config.json
model.safetensors
training_args.bin
tokenizer_config.json
tokenizer.json


In [ ]:
model.save_pretrained("/content/drive/MyDrive/muril-sms-fraud-final")
tokenizer.save_pretrained("/content/drive/MyDrive/muril-sms-fraud-final")
print("✅ Done!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Done!


In [4]:
!pip install optimum[exporters] onnx onnxruntime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 59.1 MB/s eta 0:00:00


In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load your trained model
model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/muril-sms-fraud-final")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/muril-sms-fraud-final")
model.eval()

# Dummy input for tracing
dummy = tokenizer("test message", return_tensors="pt", padding="max_length", max_length=128, truncation=True)

# Export to ONNX first
torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    "/content/muril.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch"},
        "attention_mask": {0: "batch"}
    },
    opset_version=12
)
print("✅ ONNX exported!")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/muril-sms-fraud-final'. Use `repo_type` argument if needed.